In [ ]:
# https://chatgpt.com/c/69b96d90-012c-83a4-bd0a-20a42a5cd8ff

In [54]:
import pandas as pd
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

from xgboost import XGBClassifier

from pgmpy.models import DiscreteBayesianNetwork 
from pgmpy.inference import VariableElimination

import shap
from tqdm import tqdm
import spacy    

In [ ]:
df = df[["title","source_domain","tweet_num","real"]]

df = df.dropna()

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [16]:
df = pd.read_csv("../dataset/FakeNewsNet.csv")

print(df.head())
print(df.shape)

                                               title  \
0  Kandi Burruss Explodes Over Rape Accusation on...   
1  People's Choice Awards 2018: The best red carp...   
2  Sophia Bush Sends Sweet Birthday Message to 'O...   
3  Colombian singer Maluma sparks rumours of inap...   
4  Gossip Girl 10 Years Later: How Upper East Sid...   

                                            news_url        source_domain  \
0  http://toofab.com/2017/05/08/real-housewives-a...           toofab.com   
1  https://www.today.com/style/see-people-s-choic...        www.today.com   
2  https://www.etonline.com/news/220806_sophia_bu...     www.etonline.com   
3  https://www.dailymail.co.uk/news/article-33655...  www.dailymail.co.uk   
4  https://www.zerchoo.com/entertainment/gossip-g...      www.zerchoo.com   

   tweet_num  real  
0         42     1  
1          0     1  
2         63     1  
3         20     1  
4         38     1  
(23196, 5)


In [17]:
real_count = (df["real"] == 1).sum()
fake_count = (df["real"] == 0).sum()

print("Real news:", real_count)
print("Fake news:", fake_count)

Real news: 17441
Fake news: 5755


In [18]:
real = df[df["real"] == 1]
fake = df[df["real"] == 0]

# Oversample fake
fake_oversampled = resample(
    fake,
    replace=True,            # allow duplicates
    n_samples=len(real),     # match real count
    random_state=42
)

# Combine
df = pd.concat([real, fake_oversampled])

In [19]:
df = df[["title","source_domain","tweet_num","real"]]

df = df.dropna()

In [20]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["real"]
)

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/all-roberta-large-v1",
    device=device
)

In [25]:
def generate_embeddings(texts):

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True
    )

    return np.array(embeddings)

In [27]:
X_train_embed = generate_embeddings(train_df["title"].tolist())
X_test_embed = generate_embeddings(test_df["title"].tolist())

Batches: 100%|██████████| 213/213 [00:23<00:00,  8.88it/s]


In [28]:
domain_scores = {
    "reuters.com":0.95,
    "apnews.com":0.94,
    "bbc.com":0.93,
    "thehindu.com":0.90,
    "today.com":0.88,
    "etonline.com":0.75,
    "toofab.com":0.70
}

In [29]:
def get_domain_score(domain):

    domain = domain.replace("www.","")

    return domain_scores.get(domain,0.5)

In [30]:
def build_extra_features(df):

    credibility = []
    tweets = []

    for _,row in df.iterrows():

        credibility.append(
            get_domain_score(row["source_domain"])
        )

        tweets.append(row["tweet_num"])

    return np.array([credibility,tweets]).T

In [51]:
def discretize_features(df, xgb_preds):

    credibility = []
    tweets = []

    for _,row in df.iterrows():

        cred = get_domain_score(row["source_domain"])

        credibility.append(1 if cred > 0.75 else 0)

        tweets.append(1 if row["tweet_num"] > 50 else 0)

    xgb_binary = xgb_preds

    return pd.DataFrame({
        "Credibility": credibility,
        "TweetSignal": tweets,
        "XGBPrediction": xgb_binary,
        "FakeNews": df["real"].values
    })

In [31]:
train_extra = build_extra_features(train_df)
test_extra = build_extra_features(test_df)

In [32]:
X_train = np.hstack([X_train_embed,train_extra])
X_test = np.hstack([X_test_embed,test_extra])

y_train = train_df["real"].values
y_test = test_df["real"].values

In [38]:
xgb_model = XGBClassifier(

    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,

    tree_method="hist",
    device="cuda",

    eval_metric="logloss"
)

In [39]:
xgb_model.fit(X_train,y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [40]:
preds = xgb_model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,preds))

print(classification_report(y_test,preds))

Accuracy: 0.9460055907017803
              precision    recall  f1-score   support

           0       0.92      0.97      0.95      3323
           1       0.97      0.92      0.95      3474

    accuracy                           0.95      6797
   macro avg       0.95      0.95      0.95      6797
weighted avg       0.95      0.95      0.95      6797



c:\Users\Acer\miniconda3\envs\torch_env\lib\site-packages\xgboost\core.py:751: UserWarning: [21:59:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [47]:
def predict_news(title,domain,tweet_count):

    emb = model.encode(title)

    credibility = get_domain_score(domain)

    extra = np.array([credibility,tweet_count])

    features = np.hstack([emb,extra]).reshape(1,-1)

    pred = xgb_model.predict(features)[0]

    prob = xgb_model.predict_proba(features)[0][1]

    return pred,prob

In [ ]:
title = "Government launches new 2 lakh crore farmer scheme"

domain = "reuters.com"

tweets = 200

r = predict_news(title,domain,tweets)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.15it/s]


(np.int64(1), np.float32(0.64138395))

In [56]:
bn_model = DiscreteBayesianNetwork ([
    ("Credibility","FakeNews"),
    ("TweetSignal","FakeNews"),
    ("XGBPrediction","FakeNews")
])

In [57]:
train_preds = xgb_model.predict(X_train)

bn_train_data = discretize_features(train_df, train_preds)

In [58]:
from pgmpy.estimators import MaximumLikelihoodEstimator

bn_model.fit(
    bn_train_data,
    estimator=MaximumLikelihoodEstimator
)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Credibility': 'N', 'TweetSignal': 'N', 'XGBPrediction': 'N', 'FakeNews': 'N'}


In [59]:
from pgmpy.inference import VariableElimination

bn_infer = VariableElimination(bn_model)

In [60]:
def predict_news(title,domain,tweet_count):

    emb = model.encode(title)

    credibility = get_domain_score(domain)

    extra = np.array([credibility,tweet_count])

    features = np.hstack([emb,extra]).reshape(1,-1)

    xgb_pred = xgb_model.predict(features)[0]

    cred_state = 1 if credibility > 0.75 else 0
    tweet_state = 1 if tweet_count > 50 else 0

    result = bn_infer.query(
        variables=["FakeNews"],
        evidence={
            "Credibility":cred_state,
            "TweetSignal":tweet_state,
            "XGBPrediction":xgb_pred
        }
    )

    prob_fake = result.values[1]

    return prob_fake

In [63]:
title = "Many people died in covid 19"

domain = "reuters.com"

tweets = 200

r = predict_news(title,domain,tweets)
r

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.76it/s]


np.float64(0.0)